# Lab 5 — Opis kształtu i klasyfikacja
## Cel: zapoznanie ze sposobami opisu kształtu i zastosowanie ich do klasyfikacji

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import cv2
import os
import zipfile
import urllib.request

from skimage import measure, morphology
from skimage.util import img_as_ubyte

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Pobranie zbioru danych benchmarkowych

Używamy zbioru **MPEG-7 CE-Shape-1** — 1400 binarnych obrazów kształtów, 70 klas po 20 obrazów.

In [ ]:
DATA_DIR = Path("data/shapes")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATASET_URL = "https://github.com/mruizve/mpeg7-ce-shape-1/archive/refs/heads/master.zip"
ZIP_PATH = Path("data/shapes.zip")

if not any(DATA_DIR.glob("*.png")) and not any(DATA_DIR.glob("*/*.png")):
    print("Pobieranie zbioru MPEG-7 CE-Shape-1...")
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall("data/")
    ZIP_PATH.unlink()
    
    extracted = list(Path("data").glob("mpeg7-ce-shape-1*"))
    if extracted:
        src = extracted[0]
        png_dirs = list(src.rglob("*.png"))
        if not png_dirs:
            png_dirs = list(src.rglob("*.gif"))
        for f in png_dirs:
            dest = DATA_DIR / f.name
            f.rename(dest)
        import shutil
        shutil.rmtree(src, ignore_errors=True)
    print(f"Pobrano {len(list(DATA_DIR.iterdir()))} plików")
else:
    print(f"Zbiór danych już istnieje: {len(list(DATA_DIR.rglob('*.png')) + list(DATA_DIR.rglob('*.gif')))} obrazów")

In [ ]:
# Wczytanie obrazów i etykiet
def load_dataset(data_dir):
    images = []
    labels = []
    
    for f in sorted(data_dir.iterdir()):
        if f.suffix.lower() not in ('.png', '.gif', '.bmp', '.jpg'):
            continue
        # nazwa pliku: klasa-numer.ext (np. apple-1.png)
        name = f.stem
        parts = name.rsplit('-', 1)
        if len(parts) == 2:
            label = parts[0]
        else:
            label = name
        
        img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.array(Image.open(f).convert('L'))
        if img is not None:
            images.append(img)
            labels.append(label)
    
    return images, labels

images_raw, labels_raw = load_dataset(DATA_DIR)
unique_labels = sorted(set(labels_raw))
print(f"Wczytano {len(images_raw)} obrazów, {len(unique_labels)} klas")
print(f"Klasy: {unique_labels[:10]}... (i {max(0, len(unique_labels)-10)} więcej)")

# Podgląd
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    if i < len(images_raw):
        idx = i * (len(images_raw) // 16)
        ax.imshow(images_raw[idx], cmap='gray')
        ax.set_title(labels_raw[idx], fontsize=8)
    ax.axis('off')
plt.suptitle("Przykładowe obrazy ze zbioru")
plt.tight_layout()
plt.show()

## 2. Podział na dane treningowe, walidacyjne i testowe

In [ ]:
le = LabelEncoder()
labels_encoded = le.fit_transform(labels_raw)

# 60% train, 20% val, 20% test
X_train_idx, X_temp_idx, y_train, y_temp = train_test_split(
    np.arange(len(images_raw)), labels_encoded, test_size=0.4, random_state=42, stratify=labels_encoded
)
X_val_idx, X_test_idx, y_val, y_test = train_test_split(
    X_temp_idx, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train_idx)}, Val: {len(X_val_idx)}, Test: {len(X_test_idx)}")

## 3. Centrowanie obiektów (środek ciężkości) i kadrowanie

Obliczamy środek ciężkości obiektu binarnego i kadrujemy obraz tak, aby obiekt znajdował się na środku.

In [ ]:
def binarize(img, threshold=128):
    """Binaryzacja obrazu — obiekt = białe piksele."""
    _, binary = cv2.threshold(img, threshold, 255, cv2.THRESH_BINARY)
    if np.mean(binary) > 127:
        binary = 255 - binary
    return binary

def center_and_crop(img, target_size=128, padding=10):
    """Centruje obiekt na środku ciężkości i kadruje do target_size x target_size."""
    binary = binarize(img)
    
    M = cv2.moments(binary)
    if M["m00"] == 0:
        return cv2.resize(img, (target_size, target_size))
    
    cx = int(M["m10"] / M["m00"])
    cy = int(M["m01"] / M["m00"])
    
    h, w = img.shape
    half = target_size // 2
    
    canvas = np.zeros((target_size * 3, target_size * 3), dtype=np.uint8)
    offset_y = target_size * 3 // 2 - cy
    offset_x = target_size * 3 // 2 - cx
    
    y1 = max(0, offset_y)
    y2 = min(canvas.shape[0], offset_y + h)
    x1 = max(0, offset_x)
    x2 = min(canvas.shape[1], offset_x + w)
    
    src_y1 = max(0, -offset_y)
    src_y2 = src_y1 + (y2 - y1)
    src_x1 = max(0, -offset_x)
    src_x2 = src_x1 + (x2 - x1)
    
    canvas[y1:y2, x1:x2] = img[src_y1:src_y2, src_x1:src_x2]
    
    center = target_size * 3 // 2
    cropped = canvas[center - half:center + half, center - half:center + half]
    
    return cropped

# Przetwarzanie
images_centered = [center_and_crop(img, target_size=128) for img in images_raw]

# Wizualizacja: przed i po
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    idx = i * (len(images_raw) // 6)
    axes[0, i].imshow(images_raw[idx], cmap='gray')
    axes[0, i].set_title(f"Oryginał\n{labels_raw[idx]}", fontsize=8)
    axes[0, i].axis('off')
    axes[1, i].imshow(images_centered[idx], cmap='gray')
    axes[1, i].set_title("Wycentrowany", fontsize=8)
    axes[1, i].axis('off')
plt.suptitle("Centrowanie obiektów wg środka ciężkości")
plt.tight_layout()
plt.show()

## 4. Normalizacja rozmiarów obrazów

Wszystkie obrazy skalujemy do jednolitego rozmiaru 128×128 px (już wykonane w kroku 3).

In [ ]:
TARGET_SIZE = 128
images_normalized = []

for img in images_centered:
    if img.shape != (TARGET_SIZE, TARGET_SIZE):
        img = cv2.resize(img, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_AREA)
    images_normalized.append(img)

print(f"Znormalizowano {len(images_normalized)} obrazów do rozmiaru {TARGET_SIZE}×{TARGET_SIZE}")
print(f"Kształt przykładowego obrazu: {images_normalized[0].shape}")

## 5. Proste cechy opisu kształtu

- **Pole powierzchni** (area) — liczba pikseli obiektu
- **Obwód** (perimeter) — długość konturu
- **Średnica równoważna** (equivalent diameter)
- **Bounding box** — rozmiar prostokąta otaczającego (szerokość, wysokość)

In [ ]:
def compute_simple_features(img):
    binary = binarize(img)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return np.zeros(5)
    
    cnt = max(contours, key=cv2.contourArea)
    
    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)
    eq_diameter = np.sqrt(4 * area / np.pi) if area > 0 else 0
    x, y, w, h = cv2.boundingRect(cnt)
    
    return np.array([area, perimeter, eq_diameter, w, h])

SIMPLE_FEATURE_NAMES = ['area', 'perimeter', 'eq_diameter', 'bbox_w', 'bbox_h']

simple_features = np.array([compute_simple_features(img) for img in images_normalized])
print(f"Proste cechy — kształt macierzy: {simple_features.shape}")
print(f"Nazwy cech: {SIMPLE_FEATURE_NAMES}")
print(f"Przykład (pierwszy obraz): {simple_features[0]}")

## 6. Złożone cechy opisu kształtu

- **Compactness** (zwartość) — 4π·area / perimeter²
- **Asymetria** (eccentricity) — stosunek osi elipsy dopasowanej
- **Kwadratura** (extent) — area / (bbox_w × bbox_h)
- **Najdłuższy / najkrótszy / średni promień** od centroidu do konturu

In [ ]:
def compute_complex_features(img):
    binary = binarize(img)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return np.zeros(6)
    
    cnt = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)
    
    # Compactness
    compactness = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0
    
    # Eccentricity (asymetria) — z dopasowanej elipsy
    if len(cnt) >= 5:
        ellipse = cv2.fitEllipse(cnt)
        (_, (ma, MA), _) = ellipse
        eccentricity = ma / MA if MA > 0 else 0
    else:
        eccentricity = 0
    
    # Extent (kwadratura)
    x, y, w, h = cv2.boundingRect(cnt)
    extent = area / (w * h) if w * h > 0 else 0
    
    # Promienie od centroidu
    M = cv2.moments(binary)
    if M["m00"] > 0:
        cx = M["m10"] / M["m00"]
        cy = M["m01"] / M["m00"]
    else:
        cx, cy = img.shape[1] / 2, img.shape[0] / 2
    
    pts = cnt.reshape(-1, 2).astype(float)
    radii = np.sqrt((pts[:, 0] - cx)**2 + (pts[:, 1] - cy)**2)
    max_radius = radii.max() if len(radii) > 0 else 0
    min_radius = radii.min() if len(radii) > 0 else 0
    mean_radius = radii.mean() if len(radii) > 0 else 0
    
    return np.array([compactness, eccentricity, extent, max_radius, min_radius, mean_radius])

COMPLEX_FEATURE_NAMES = ['compactness', 'eccentricity', 'extent', 'max_radius', 'min_radius', 'mean_radius']

complex_features = np.array([compute_complex_features(img) for img in images_normalized])
print(f"Złożone cechy — kształt macierzy: {complex_features.shape}")
print(f"Nazwy cech: {COMPLEX_FEATURE_NAMES}")
print(f"Przykład (pierwszy obraz): {complex_features[0]}")

## 7–9. Klasyfikacja z prostymi cechami

Łączymy proste i złożone cechy, normalizujemy i klasyfikujemy za pomocą kNN, drzewa decyzyjnego i regresji logistycznej.

In [ ]:
# Połączenie wszystkich prostych + złożonych cech
all_simple_complex = np.hstack([simple_features, complex_features])
ALL_FEATURE_NAMES = SIMPLE_FEATURE_NAMES + COMPLEX_FEATURE_NAMES

# Podział wg wcześniejszych indeksów
X_train_sc = all_simple_complex[X_train_idx]
X_val_sc = all_simple_complex[X_val_idx]
X_test_sc = all_simple_complex[X_test_idx]

# Normalizacja
scaler_sc = StandardScaler()
X_train_sc = scaler_sc.fit_transform(X_train_sc)
X_val_sc = scaler_sc.transform(X_val_sc)
X_test_sc = scaler_sc.transform(X_test_sc)

print(f"Cechy: {ALL_FEATURE_NAMES}")
print(f"Kształt X_train: {X_train_sc.shape}")

In [ ]:
def evaluate_classifier(clf, name, X_tr, y_tr, X_v, y_v, X_te, y_te):
    clf.fit(X_tr, y_tr)
    
    y_val_pred = clf.predict(X_v)
    y_test_pred = clf.predict(X_te)
    
    val_acc = accuracy_score(y_v, y_val_pred)
    test_acc = accuracy_score(y_te, y_test_pred)
    
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"Val accuracy:  {val_acc:.4f}")
    print(f"Test accuracy: {test_acc:.4f}")
    print(f"\nClassification Report (test):")
    print(classification_report(y_te, y_test_pred, zero_division=0, output_dict=False)[:500])
    
    return y_test_pred, test_acc

classifiers = {
    'kNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=20),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
}

results_simple = {}
predictions_simple = {}

for name, clf in classifiers.items():
    pred, acc = evaluate_classifier(clf, name, X_train_sc, y_train, X_val_sc, y_val, X_test_sc, y_test)
    results_simple[name] = acc
    predictions_simple[name] = pred

In [ ]:
# Confusion matrix dla najlepszego klasyfikatora
best_name = max(results_simple, key=results_simple.get)
best_pred = predictions_simple[best_name]

fig, ax = plt.subplots(figsize=(14, 12))
cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=90, values_format='d')
ax.set_title(f"Confusion Matrix — {best_name} (proste+złożone cechy)\nAccuracy: {results_simple[best_name]:.4f}")
plt.tight_layout()
plt.show()

## 10. Zaawansowane deskryptory — Deskryptor Fouriera + Momenty Zernike'a

### Deskryptor Fouriera (Fourier Descriptor)
Kontur obiektu traktujemy jako sygnał zespolony z(t) = x(t) + j·y(t). Transformata Fouriera tego sygnału daje współczynniki opisujące kształt niezależnie od translacji, skali i rotacji (po odpowiedniej normalizacji).

### Momenty Zernike'a
Momenty Zernike'a to projekcje obrazu na bazę wielomianów ortogonalnych Zernike'a na dysku jednostkowym. Są z natury niezmiennicze względem rotacji (moduły momentów) i dobrze opisują złożone kształty.

In [ ]:
def fourier_descriptor(img, n_coeffs=32):
    """Oblicza znormalizowany deskryptor Fouriera konturu."""
    binary = binarize(img)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    
    if not contours:
        return np.zeros(n_coeffs)
    
    cnt = max(contours, key=cv2.contourArea)
    pts = cnt.reshape(-1, 2).astype(np.float64)
    
    # Sygnał zespolony
    z = pts[:, 0] + 1j * pts[:, 1]
    
    # DFT
    coeffs = np.fft.fft(z)
    
    # Normalizacja: niezmienniczość translacji (usunięcie DC), skali (dzielenie przez |c1|), rotacji (moduły)
    coeffs = coeffs[1:]  # usuwamy DC (translacja)
    if np.abs(coeffs[0]) > 0:
        coeffs = coeffs / np.abs(coeffs[0])  # normalizacja skali
    
    fd = np.abs(coeffs[:n_coeffs])  # moduły = niezmienniczość rotacji
    
    if len(fd) < n_coeffs:
        fd = np.pad(fd, (0, n_coeffs - len(fd)))
    
    return fd

# Test
fd_example = fourier_descriptor(images_normalized[0])
print(f"Deskryptor Fouriera — {len(fd_example)} współczynników")

plt.figure(figsize=(10, 3))
plt.bar(range(len(fd_example)), fd_example)
plt.xlabel("Numer współczynnika")
plt.ylabel("|F(k)|")
plt.title("Deskryptor Fouriera — przykładowy obraz")
plt.show()

In [ ]:
def zernike_radial(n, m, rho):
    """Wielomian radialny Zernike'a R_nm(rho)."""
    R = np.zeros_like(rho)
    m_abs = abs(m)
    for s in range((n - m_abs) // 2 + 1):
        num = ((-1) ** s) * np.math.factorial(n - s)
        den = (np.math.factorial(s) *
               np.math.factorial((n + m_abs) // 2 - s) *
               np.math.factorial((n - m_abs) // 2 - s))
        R += (num / den) * rho ** (n - 2 * s)
    return R

def zernike_moments(img, max_order=12):
    """Oblicza momenty Zernike'a do zadanego rzędu."""
    binary = binarize(img).astype(np.float64) / 255.0
    
    h, w = binary.shape
    y, x = np.mgrid[0:h, 0:w]
    
    # Mapowanie na dysk jednostkowy
    cx, cy = w / 2, h / 2
    radius = max(cx, cy)
    x_norm = (x - cx) / radius
    y_norm = (y - cy) / radius
    
    rho = np.sqrt(x_norm**2 + y_norm**2)
    theta = np.arctan2(y_norm, x_norm)
    
    mask = rho <= 1.0
    
    moments = []
    for n in range(max_order + 1):
        for m in range(-n, n + 1, 2):
            if (n - abs(m)) % 2 != 0:
                continue
            R = zernike_radial(n, m, rho)
            V = R * np.exp(-1j * m * theta)
            Z = ((n + 1) / np.pi) * np.sum(binary[mask] * np.conj(V[mask]))
            moments.append(np.abs(Z))
    
    return np.array(moments)

# Test
zm_example = zernike_moments(images_normalized[0], max_order=12)
print(f"Momenty Zernike'a — {len(zm_example)} momentów (do rzędu 12)")

plt.figure(figsize=(10, 3))
plt.bar(range(len(zm_example)), zm_example)
plt.xlabel("Numer momentu")
plt.ylabel("|Z_nm|")
plt.title("Momenty Zernike'a — przykładowy obraz")
plt.show()

In [ ]:
# Obliczenie zaawansowanych deskryptorów dla wszystkich obrazów
print("Obliczanie deskryptorów Fouriera...")
fourier_features = np.array([fourier_descriptor(img) for img in images_normalized])
print(f"  Kształt: {fourier_features.shape}")

print("Obliczanie momentów Zernike'a...")
zernike_features = np.array([zernike_moments(img) for img in images_normalized])
print(f"  Kształt: {zernike_features.shape}")

## 12–13. Klasyfikacja z zaawansowanymi deskryptorami i porównanie wyników

Porównujemy:
1. Tylko deskryptor Fouriera
2. Tylko momenty Zernike'a
3. Proste cechy + deskryptor Fouriera
4. Proste cechy + momenty Zernike'a
5. Wszystkie cechy razem

In [ ]:
feature_sets = {
    'Proste + złożone cechy': all_simple_complex,
    'Fourier Descriptor (sam)': fourier_features,
    'Zernike Moments (sam)': zernike_features,
    'Proste + Fourier': np.hstack([all_simple_complex, fourier_features]),
    'Proste + Zernike': np.hstack([all_simple_complex, zernike_features]),
    'Wszystkie cechy': np.hstack([all_simple_complex, fourier_features, zernike_features]),
}

results_all = {}

for feat_name, features in feature_sets.items():
    X_tr = features[X_train_idx]
    X_v = features[X_val_idx]
    X_te = features[X_test_idx]
    
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_v = scaler.transform(X_v)
    X_te = scaler.transform(X_te)
    
    # Używamy kNN jako klasyfikatora referencyjnego
    clf = KNeighborsClassifier(n_neighbors=5)
    clf.fit(X_tr, y_train)
    
    val_acc = accuracy_score(y_val, clf.predict(X_v))
    test_acc = accuracy_score(y_test, clf.predict(X_te))
    
    results_all[feat_name] = {'val': val_acc, 'test': test_acc, 'n_features': features.shape[1]}
    print(f"{feat_name:30s} | {features.shape[1]:3d} cech | Val: {val_acc:.4f} | Test: {test_acc:.4f}")

In [ ]:
# Wizualizacja porównania
names = list(results_all.keys())
test_accs = [results_all[n]['test'] for n in names]
val_accs = [results_all[n]['val'] for n in names]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(names))
width = 0.35

bars1 = ax.bar(x - width/2, val_accs, width, label='Val accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, test_accs, width, label='Test accuracy', color='coral')

ax.set_ylabel('Accuracy')
ax.set_title('Porównanie zestawów cech — klasyfikacja kNN (k=5)')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax.legend()
ax.set_ylim(0, 1)

for bar in bars1 + bars2:
    h = bar.get_height()
    ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix — najlepsza kombinacja cech z kNN
best_feat_name = max(results_all, key=lambda k: results_all[k]['test'])
best_features = feature_sets[best_feat_name]

scaler_best = StandardScaler()
X_tr_best = scaler_best.fit_transform(best_features[X_train_idx])
X_te_best = scaler_best.transform(best_features[X_test_idx])

clf_best = KNeighborsClassifier(n_neighbors=5)
clf_best.fit(X_tr_best, y_train)
y_pred_best = clf_best.predict(X_te_best)

fig, ax = plt.subplots(figsize=(14, 12))
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=90, values_format='d')
ax.set_title(f"Confusion Matrix — kNN + {best_feat_name}\nAccuracy: {results_all[best_feat_name]['test']:.4f}")
plt.tight_layout()
plt.show()

print(f"\nNajlepsza kombinacja: {best_feat_name}")
print(f"Test accuracy: {results_all[best_feat_name]['test']:.4f}")

## Podsumowanie

| Zestaw cech | Opis |
|---|---|
| Proste + złożone | area, perimeter, eq_diameter, bbox, compactness, eccentricity, extent, promienie |
| Fourier Descriptor | 32 współczynniki znormalizowanej DFT konturu |
| Zernike Moments | momenty Zernike'a do 12. rzędu |
| Kombinacje | połączenia prostych cech z zaawansowanymi deskryptorami |

Zaawansowane deskryptory (Fourier, Zernike) znacząco poprawiają klasyfikację kształtów w porównaniu z samymi prostymi cechami geometrycznymi, ponieważ kodują globalną strukturę konturu/kształtu.